# ThirdUmpire — Reranker Fine-tuning (PROJECT_PLAN.md Phase 6)

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-6-v2` on synthetic (query, passage, label) pairs mined from the ThirdUmpire corpus, with the real golden questions excluded (leakage guard already applied in `train/build_dataset.py`).

**Before running:** on `Runtime > Change runtime type`, select a **GPU** (a free T4 is enough for this model/dataset size).

**What you need to upload** (produced locally by `python -m train.build_dataset`):
- `train/data/train.jsonl`
- `train/data/val.jsonl`
- `train/data/dataset_info.json`

**What you'll get back:** a `reranker-ft.zip` download — unzip it into `models/reranker-ft/` in the project (that's `RERANKER_FT_PATH` in `src/config.py`).

In [ ]:
!pip install -q -U "sentence-transformers>=4" datasets accelerate

In [ ]:
import torch
import sentence_transformers

print("sentence-transformers:", sentence_transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- go to Runtime > Change runtime type and select a GPU, then re-run this cell.")

# PROJECT_PLAN.md Phase 6 says to check the installed version and use the
# matching CrossEncoder training API: v4+ uses CrossEncoderTrainer +
# BinaryCrossEntropyLoss (what this notebook uses throughout). If Colab
# ever installs something older than 4.x, stop here and consult the
# sentence-transformers CrossEncoder migration guide before continuing --
# the older `CrossEncoder.fit()` API has a different shape than the cells
# below.
major_version = int(sentence_transformers.__version__.split(".")[0])
assert major_version >= 4, (
    f"sentence-transformers {sentence_transformers.__version__} is older than the v4+ CrossEncoderTrainer API "
    "this notebook uses. Upgrade (see the pip install cell above) or adapt the training cells below."
)

In [ ]:
from google.colab import files

print("Select train.jsonl, val.jsonl and dataset_info.json (from train/data/ in the project):")
uploaded = files.upload()
assert "train.jsonl" in uploaded, "train.jsonl was not uploaded"
assert "val.jsonl" in uploaded, "val.jsonl was not uploaded"
assert "dataset_info.json" in uploaded, "dataset_info.json was not uploaded"

In [ ]:
import json

from datasets import Dataset


def load_rows(path: str) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


train_rows = load_rows("train.jsonl")
val_rows = load_rows("val.jsonl")

with open("dataset_info.json", "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

print("dataset_info.json:", json.dumps(dataset_info, indent=2))
print(f"train rows: {len(train_rows)}, val rows: {len(val_rows)}")

# Column order matters for BinaryCrossEntropyLoss: the first text column is
# the (anchor) query, the second is the candidate passage.
train_dataset = Dataset.from_dict(
    {
        "query": [r["query"] for r in train_rows],
        "passage": [r["passage"] for r in train_rows],
        "label": [r["label"] for r in train_rows],
    }
)
print(train_dataset)

In [ ]:
# CrossEncoderRerankingEvaluator needs {"query": ..., "positive": [...], "negative": [...]}
# per query, not flat (query, passage, label) rows -- group val_rows back up.
from collections import defaultdict

by_query = defaultdict(lambda: {"positive": [], "negative": []})
for row in val_rows:
    bucket = "positive" if row["label"] == 1 else "negative"
    by_query[row["query"]][bucket].append(row["passage"])

val_samples = [
    {"query": query, "positive": buckets["positive"], "negative": buckets["negative"]}
    for query, buckets in by_query.items()
    if buckets["positive"] and buckets["negative"]  # the evaluator needs both to rank
]
print(f"{len(val_samples)} evaluator queries (of {len(by_query)} distinct val queries)")

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

# Mirrors src/config.py's RERANKER_MODEL and RERANKER_FT_* defaults -- keep
# these in sync if that file changes.
BASE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_LENGTH = 512
EPOCHS = 2
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
WARMUP_RATIO = 0.1
AT_K = 10  # MRR@10 / NDCG@10

model = CrossEncoder(BASE_MODEL, num_labels=1, max_length=MAX_LENGTH)
loss = BinaryCrossEntropyLoss(model)

evaluator = CrossEncoderRerankingEvaluator(
    samples=val_samples,
    at_k=AT_K,
    name="val",
    show_progress_bar=True,
)

# Baseline (pre-fine-tuning) numbers, for an honest before/after comparison
# in README.md later -- report these even if the fine-tuned model doesn't
# win (PROJECT_PLAN.md Phase 6 "Done when").
baseline_metrics = evaluator(model)
print("Baseline (before fine-tuning):", json.dumps(baseline_metrics, indent=2))

In [ ]:
# The evaluator's raw metric keys ("mrr@10", "ndcg@10", ...) get prefixed
# with the evaluator's `name` ("val_mrr@10"), and the Trainer prefixes
# that again with "eval_" when logging -- metric_for_best_model is given
# WITHOUT that "eval_" prefix per the standard HF Trainer convention.
args = CrossEncoderTrainingArguments(
    output_dir="reranker-ft-checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model=f"val_mrr@{AT_K}",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = CrossEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=evaluator,
)
trainer.train()

In [ ]:
final_metrics = evaluator(model)
print("Baseline:", json.dumps(baseline_metrics, indent=2))
print("Fine-tuned (best checkpoint):", json.dumps(final_metrics, indent=2))
print(
    f"MRR@{AT_K}: {baseline_metrics[f'val_mrr@{AT_K}']:.4f} -> {final_metrics[f'val_mrr@{AT_K}']:.4f}"
)

# Report these numbers honestly in README.md even if the fine-tuned model
# doesn't beat the baseline (PROJECT_PLAN.md Phase 6 "Done when").

In [ ]:
import shutil

SAVE_DIR = "reranker-ft"
model.save_pretrained(SAVE_DIR)

# Carry the dataset provenance along with the model so it's traceable to
# exactly which corpus/first-stage it was trained on.
with open(f"{SAVE_DIR}/dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)
with open(f"{SAVE_DIR}/eval_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"baseline": baseline_metrics, "fine_tuned": final_metrics}, f, indent=2)

shutil.make_archive("reranker-ft", "zip", root_dir=".", base_dir=SAVE_DIR)
files.download("reranker-ft.zip")

print("Downloaded reranker-ft.zip -- unzip it so its contents land at models/reranker-ft/ in the project.")